In [23]:
import os, re, time
import pandas as pd
import pytz
from datetime import datetime
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# 🔧 1. การตั้งค่าพื้นฐาน
# ==========================================
thailand_tz = pytz.timezone("Asia/Bangkok")
reference_date = datetime.now(thailand_tz).strftime("%d/%m/%y")

konvy_email = ""
konvy_password = ""
product_type_list = ["Mizumi", "Sentle Mide", "Srichand", "alese"] 
output_file = rf"D:\konvy_data_full.xlsx"

# เตรียม Browser
options = webdriver.ChromeOptions()
# options.add_argument("--headless") # ปิดไว้ก่อนเพื่อดูว่ามันกวาดลิงก์ได้จริงไหม
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 20)

def smart_click(locator):
    try:
        WebDriverWait(driver, 5).until(EC.invisibility_of_element_located((By.CLASS_NAME, "waf-nc-mask")))
    except: pass
    element = wait.until(EC.element_to_be_clickable(locator))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", element)

# ==========================================
# 🔍 2. Phase 1: กวาดลิงก์ด้วย Selenium (แก้ปัญหา Found 0)
# ==========================================
all_product_links = []
print("🚀 Phase 1: Collecting product links with Selenium...")

for product_type in product_type_list:
    url = f"https://www.konvy.com/list/?title={product_type}"
    print(f"🔎 Scanning brand: {product_type}")
    driver.get(url)
    time.sleep(3) # รอหน้าโหลดเบื้องต้น
    
    # เช็คจำนวนหน้า
    try:
        page_nav = driver.find_elements(By.CSS_SELECTOR, "div.sc_paging a")
        page_nums = [int(p.text) for p in page_nav if p.text.isdigit()]
        max_page = max(page_nums) if page_nums else 1
    except:
        max_page = 1

    for x in range(1, max_page + 1):
        if x > 1:
            driver.get(f"https://www.konvy.com/list/?title={product_type}&page={x}")
            time.sleep(2)
        
        # เลื่อนหน้าจอลงเพื่อให้สินค้าโหลด (Lazy Load)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        
        soup = BeautifulSoup(driver.page_source, "lxml")
        # หากล่องสินค้าโดยเน้นที่ class 'product-name' เป็นหลัก
        product_names = soup.find_all("p", class_="product-name")
        
        count_page = 0
        for name_tag in product_names:
            parent = name_tag.find_parent("div")
            if not parent: continue
            
            link_tag = parent.find("a", href=True)
            img_tag = parent.find("img")
            
            if link_tag and "/products/" in link_tag['href']:
                href = link_tag["href"]
                full_link = href if href.startswith("http") else f"https:{href}"
                img_url = img_tag.get("original") or img_tag.get("src") or ""
                
                all_product_links.append({
                    "product_type": product_type,
                    "brand": product_type,
                    "link": full_link,
                    "image": img_url
                })
                count_page += 1
        print(f"   ✅ Page {x}: Found {count_page} products.")

# ลบลิงก์ซ้ำ
df_links = pd.DataFrame(all_product_links).drop_duplicates(subset=['link'])
product_links = df_links.to_dict('records')
print(f"🔗 Total unique links: {len(product_links)}")


# ==========================================
# 🔑 3. Login (ปรับปรุงใหม่เพื่อแก้ Timeout)
# ==========================================
print("\n🔑 Step: Logging in...")

# วนลูปพยายามเข้าหน้า Login (เผื่อหน้าขาว)
login_success = False
for retry in range(3):
    try:
        driver.get("https://www.konvy.com/account/login.php")
        # รอให้ Body ของเว็บปรากฏขึ้นมาจริงๆ
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        
        # 🍪 ปิด Cookie (ถ้ามี)
        try:
            cookie_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.got-it, .accept-cookie")))
            driver.execute_script("arguments[0].click();", cookie_btn)
            print("🍪 Cookie closed.")
        except: pass

        # 📧 คลิกสลับโหมด Login (ใช้ XPath ที่เจาะจงตำแหน่งแทน Class อย่างเดียว)
        print(f"📧 Attempting to switch to Email Login (Retry {retry+1})...")
        
        # ลองหาจากหลาย Selector: 1. Class 2. Text 3. Position
        selectors = [
            (By.CLASS_NAME, "password-icon"),
            (By.XPATH, "//div[contains(text(), 'เข้าสู่ระบบด้วยรหัสผ่าน')]"),
            (By.XPATH, "//div[@class='login-type-con']/div[2]")
        ]
        
        target_tab = None
        for s in selectors:
            try:
                target_tab = wait.until(EC.element_to_be_clickable(s))
                if target_tab: break
            except: continue
        
        if target_tab:
            driver.execute_script("arguments[0].click();", target_tab)
            time.sleep(2) # รอให้ Form กางออก
            
            # กรอก Email ในฟอร์มรหัสผ่าน
            print("⌨️ Entering credentials...")
            email_xpath = "//form[@name='login_form_pwd']//input[contains(@placeholder, 'เมล') or @type='text']"
            email_field = wait.until(EC.visibility_of_element_located((By.XPATH, email_xpath)))
            email_field.clear()
            email_field.send_keys(konvy_email)

            pass_field = driver.find_element(By.XPATH, "//form[@name='login_form_pwd']//input[@type='password']")
            pass_field.clear()
            pass_field.send_keys(konvy_password)

            # คลิกปุ่ม Login
            login_btn = driver.find_element(By.CSS_SELECTOR, "form[name='login_form_pwd'] a.btn")
            driver.execute_script("arguments[0].click();", login_btn)
            
            print("✅ Login process sent.")
            login_success = True
            break
    except Exception as e:
        print(f"⚠️ Login attempt {retry+1} failed. Retrying...")
        time.sleep(3)

if not login_success:
    print("❌ Failed to login after multiple attempts.")
    driver.save_screenshot("login_timeout_debug.png")


# ==========================================
# 📊 4. Phase 2: Scrape รายละเอียดสินค้า
# ==========================================
# เช็คข้อมูลเดิมเพื่อข้าม (Skip Logic)
scraped_urls = set()
if os.path.exists(output_file):
    try:
        df_old = pd.read_excel(output_file)
        scraped_urls = set(df_old[df_old["Reference_date"] == reference_date]["Product Url"].tolist())
    except: pass

todo_links = [p for p in product_links if p["link"] not in scraped_urls]
print(f"🎯 Net items to scrape: {len(todo_links)}")

skincare_list = []
for i, item in enumerate(todo_links):
    print(f"📦 [{i+1}/{len(todo_links)}] Scrapping: {item['link']}")
    driver.get(item["link"])
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "lxml")

    res = {
        "Product Name": soup.title.text.split("-")[0].strip() if soup.title else "N/A",
        "Brand": item["brand"],
        "Image Url": item["image"],
        "Product Url": item["link"],
        "Reference_date": reference_date,
        "Product Type": item["product_type"]
    }

    price_tag = soup.select_one(".pro-top-con-price .ky-fw-500, .ky-fw-500")
    res["Sell Price"] = price_tag.text.strip() if price_tag else "N/A"

    attr_box = soup.find("div", class_="attribute-box")
    if attr_box:
        for li in attr_box.find_all("li"):
            lbl, val = li.find("span", class_="left"), li.find("span", class_="sub_attribute") or li.find("a")
            if lbl and val: res[lbl.text.strip()] = val.text.strip()

    skincare_list.append(res)

driver.quit()

# ==========================================
# 💾 5. Save Results
# ==========================================
if skincare_list:
    df_new = pd.DataFrame(skincare_list)
    if os.path.exists(output_file):
        df_final = pd.concat([pd.read_excel(output_file), df_new], ignore_index=True)
        df_final.drop_duplicates(subset=["Reference_date", "Product Url"], keep='last', inplace=True)
        df_final.to_excel(output_file, index=False)
    else:
        df_new.to_excel(output_file, index=False)
    print(f"🎉 Success! Data saved to {output_file}")

🚀 Phase 1: Collecting product links with Selenium...
🔎 Scanning brand: Mizumi
   ✅ Page 1: Found 0 products.
🔎 Scanning brand: Sentle Mide
   ✅ Page 1: Found 0 products.
🔎 Scanning brand: Srichand
   ✅ Page 1: Found 0 products.
🔎 Scanning brand: alese
   ✅ Page 1: Found 0 products.
🔗 Total unique links: 0

🔑 Step: Logging in...
📧 Attempting to switch to Email Login (Retry 1)...
📧 Attempting to switch to Email Login (Retry 2)...
📧 Attempting to switch to Email Login (Retry 3)...
❌ Failed to login after multiple attempts.
🎯 Net items to scrape: 0
